# Milestone 1 — arXiv crawler on Colab
Runtime → Change runtime type → **CPU** (the benchmark testbed). Run cells in order.

In [ ]:
# 1. Get the code (replace with YOUR group repo) and install requirements
!git clone https://github.com/YOUR-ORG/YOUR-CODE-REPO.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!pip install -q -r src/requirements.txt

In [ ]:
# 2. Mount Drive (backup/restore, because Colab sessions can disconnect)
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/arxiv_backup'
!mkdir -p $BACKUP
# Restore previous progress (no-op on the first run)
!rsync -a $BACKUP/corpus/ /content/corpus/ 2>/dev/null; rsync -a $BACKUP/work/ /content/work/ 2>/dev/null; echo restored

In [ ]:
# 3. Semantic Scholar key from Colab Secrets (sidebar key icon -> add secret S2_API_KEY, enable notebook access).
# IMPORTANT: copy it into an environment variable, because `!python src/main.py` runs in a
# child process that cannot read Colab Secrets directly.
import os
from google.colab import userdata
try:
    os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY").strip()
    print("S2 key exported to the environment, length:", len(os.environ["S2_API_KEY"]))
except Exception as error:
    print("No key -> slower reference crawl (3.1 s/request):", error)
!python -c "import os; print('child process sees key:', bool(os.environ.get('S2_API_KEY')))"

In [ ]:
# 4. Smoke test on 5 papers, in a separate folder (check outputs before the long run)
!python src/main.py --range 2605.02338 2605.07337 --limit 5 --corpus-dir /content/test_corpus --work-dir /content/test_work
!cat /content/test_work/metrics.json
!head -n 5 /content/test_work/papers_log.csv

In [ ]:
# 5. Background backup to Drive every 10 minutes (keep this cell running)
import threading, subprocess, time
def backup_loop():
    while True:
        time.sleep(600)
        subprocess.run(f"rsync -a /content/corpus/ {BACKUP}/corpus/; rsync -a /content/work/ {BACKUP}/work/", shell=True)
threading.Thread(target=backup_loop, daemon=True).start()

In [ ]:
# 6. FULL RUN of the WHOLE GROUP (group_ids.txt: one id or range A..B per line, in the repo root).
# Wrapped in /usr/bin/time -v for an independent peak-RAM figure. Retries and cooldowns count as runtime.
# The folders must exist BEFORE the shell opens the redirect target below.
!mkdir -p /content/corpus /content/work
!which time > /dev/null || apt-get -qq install -y time
# 2>&1 | tee: show the live log AND keep it (with the time -v report) in time_v.txt
!/usr/bin/time -v python src/main.py --ids-file group_ids.txt --corpus-dir /content/corpus --work-dir /content/work 2>&1 | tee /content/work/time_v.txt
!grep -E "Maximum resident|Elapsed" /content/work/time_v.txt

In [ ]:
# 7. Final backup + quick audit
!rsync -a /content/corpus/ $BACKUP/corpus/ && rsync -a /content/work/ $BACKUP/work/
!du -sh /content/corpus; ls /content/corpus | head
!cat /content/work/metrics.json
!python -c "import json; m=json.load(open('/content/work/metrics.json')); print({k:v for k,v in m.items() if k.startswith(('arxiv_','s2_'))})"

In [ ]:
# 8. Download the small files needed by the report (copy them to book/data/ in the report repo)
!cd /content/work && zip -j /content/report_data.zip metrics.json papers_log.csv versions_log.csv time_v.txt crawl.log
from google.colab import files
files.download('/content/report_data.zip')

**If the session disconnects or the run stops with exit code 1:** run cells 1, 2 (restores progress), 3, 5, 6 again — finished items are skipped. Do **not** commit `corpus/` or `work/` to git.